This is a first attempt at classifying the AppML_ZTF_table.csv dataset, in order to find anomalies through classifying outliers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import functions_io as io

from sklearn.metrics import adjusted_rand_score, silhouette_score

# import functions_classification as clf


In [ ]:
# read in data and prep
data = io.read_csv()
data = io.remove_columns(data)
X = io.impute_and_scale(data)


In [ ]:
print(X.shape)
from sklearn.decomposition import PCA

# PCA to check variance explained
pca = PCA().fit(X)
cumvar = np.cumsum(pca.explained_variance_ratio_)
# Find n_components for ~99% variance
n_components_99 = np.argmax(cumvar >= 0.99) + 1
print(f"Number of PCA components for 99% variance: {n_components_99}")

Start by breaking it all down into 2d using UMAP

In [ ]:
import umap 

# UMAP for visualization and as clustering input
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
# reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.2)
embedding = reducer.fit_transform(X)

sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1])
plt.show()

In [ ]:
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=30, metric="euclidean")
labels = clusterer.fit_predict(embedding)

print('Clustered into {} clusters.'.format(len(set(labels)) - (1 if -1 in labels else 0)))

# -1 = noise points in HDBSCAN
# print(pd.Series(labels).value_counts())

sil = silhouette_score(embedding, labels)
print("Silhouette Score:", sil)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=labels)
plt.title(f"HDBSCAN Clusters in UMAP Space\nSilhouette: {sil:.2f}")
plt.legend(title="Cluster")
plt.show()


In [ ]:
# get anomalies from outlierscore
outlierscore = clusterer.outlier_scores_
# get points with outlierscore > 0.75
anomalies = embedding[outlierscore > 0.75]
print(f"Number of anomalies (outlier score > 0.75): {len(anomalies)}")
# plot all points colored by outlier score, with a colorbar going from 0.75 to 1.0
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.scatterplot(x=anomalies[:, 0], y=anomalies[:, 1], hue=outlierscore[outlierscore > 0.75], palette="viridis", legend=True)
# plt.colorbar(outlierscore[outlierscore > 0.75], cax=ax, label='HDBSCAN Outlier Score')
# ax.title("HDBSCAN Outlier Scores in UMAP Space")
plt.show()

We see a bunch of unclassified objects here. This is a great sign, as that would mean they could be actual anomalies :)

In [ ]:
# check if any of these outliers have labels available
classifications = io.read_get_classification_labels(type='tns_type')
print(classifications[outlierscore > 0.75])
# print(np.unique(classifications[outlierscore > 0.75]))

# and get their object ids
object_ids = io.get_object_ids(idxs=np.where(outlierscore > 0.75)[0])
np.savetxt("../outputs/potential_anomalies_ids.txt", np.array([object_ids, classifications[outlierscore > 0.75]]).T, fmt="%s")